# 3 - Vector layers

The vector builders read a GeoDataFrame (or a pyramids `FeatureCollection`): `points`, `lines`, `polygons`,
`choropleth` (polygons coloured by a value), `labels` / `text` (annotation), the vector-field family
(`vectorfield`, `streamlines`) and `graph`.

**Setup** and data - the German states, Heidelberg bird points and roads, and a gradient field from the
population raster.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # hide backend-mismatch style warnings in the docs

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image, display

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import geopandas as gpd
import numpy as np
from pyramids.dataset import GeoReference

birds = gpd.read_file(str(DATA / "heidelberg_birds_2023.geojson"))
birds["log_uncertainty"] = np.log10(birds["uncertainty_m"].clip(lower=1.0))
roads = gpd.read_file(str(DATA / "heidelberg_roads.geojson"))
adm1 = gpd.read_file(str(DATA / "germany_adm1.geojson"))
adm1["area_km2"] = adm1.to_crs(3035).area / 1e6

temp = Dataset.read_file(str(DATA / "global" / "ncep_tavg_07.tif"))  # small 72x144 grid for vector fields
ta = temp.read_array(band=0).astype("float32")
gy, gx = np.gradient(ta)  # down-gradient (negative) points from warm toward cold; coarse grid keeps it cheap
# Wrap each component back into a Dataset that shares temp's geotransform + CRS, so vectorfield / streamlines
# place the arrows in lon/lat (EPSG:4326). A bare NumPy array would land on a 0..143 / 0..71 pixel-index grid.
_gref = GeoReference(geo=temp.geotransform, epsg=temp.epsg)
u = Dataset.from_array(arr=-gx, geo_ref=_gref, no_data_value=np.nan)
v = Dataset.from_array(arr=-gy, geo_ref=_gref, no_data_value=np.nan)
print("birds:", len(birds), "| roads:", len(roads), "| states:", len(adm1), "| temp grid:", ta.shape)

## `points` - a point layer coloured by a value

`column=` colours the markers; `size` and `cmap` style them, and `colorbar()` adds the guide.

In [ ]:
m = InteractiveMap(crs=4326, title="Heidelberg birds by log10 location uncertainty (m)")
m.points(birds, column="log_uncertainty", size=6, cmap="viridis").colorbar()
show(m)

## `lines` - a line layer

`lines` draws a line/MultiLineString layer; here the Heidelberg road network.

In [ ]:
m = InteractiveMap(crs=4326, title="Heidelberg road network")
m.lines(roads, color="steelblue", line_width=1)
show(m)

## `polygons` - a polygon layer

In [ ]:
m = InteractiveMap(crs=4326, title="German federal states")
m.polygons(adm1, line_color="black", fill_alpha=0.1)
show(m)

## `choropleth` - polygons coloured by a value

`choropleth` fills polygons by a column - here each state by its area.

In [ ]:
m = InteractiveMap(crs=4326, title="German states by area (km2)")
m.choropleth(adm1, "area_km2", cmap="viridis").colorbar()
show(m)

## `labels` and `text`

`labels` draws one label per feature from a column; `text` places a single caption at a coordinate.

In [ ]:
m = InteractiveMap(crs=4326, title="labelled states")
m.polygons(adm1, line_color="grey", fill_alpha=0.05)
m.labels(adm1.assign(geometry=adm1.representative_point()), "NAME_1" if "NAME_1" in adm1 else adm1.columns[0],
         text_font_size="7pt")
m.text(10.4, 48.2, "Bavaria", text_font_size="11pt", text_color="navy")
show(m)

## Vector fields - `vectorfield` and `streamlines`

Given two component arrays `u`, `v` on a grid, the backend draws the flow. `vectorfield` has a Bokeh glyph
(interactive); `streamlines` is matplotlib-only, so `show(static=True)` embeds a PNG.

In [ ]:
m = InteractiveMap(crs=temp.epsg, title="July temperature gradient (vectorfield)")
m.vectorfield(u, v, density=0.5, cmap="viridis")
show(m)

In [ ]:
m = InteractiveMap(crs=temp.epsg, title="streamlines")
m.streamlines(u, v, density=0.6)
show(m, static=True)

## `graph` - a node/edge network

`graph` draws a network from a nodes GeoDataFrame and a list of `(from, to, weight)` edges.

In [ ]:
nodes = gpd.GeoDataFrame(
    {"id": [0, 1, 2, 3]},
    geometry=gpd.points_from_xy([8.6, 8.7, 8.8, 8.7], [49.35, 49.42, 49.40, 49.50]),
    crs="EPSG:4326",
)
edges = [(0, 1, 5.0), (1, 2, 3.0), (1, 3, 2.0)]
m = InteractiveMap(crs=4326, title="a small network (graph)")
m.graph(nodes, edges, weight="weight")
show(m)